In [7]:
import pandas as pd

resale = pd.read_csv("data/resale_prices.csv")
print(resale.columns)

rpi = pd.read_csv("raw/HousingAndDevelopmentBoardHDBResalePriceIndex1Q2009100Quarterly.csv")
print(rpi.columns)

Index(['Unnamed: 0', 'month', 'town', 'flat_type', 'street_name',
       'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date',
       'resale_price', 'remaining_lease', 'year', 'avg_storey'],
      dtype='object')
Index(['DataSeries', '20254Q', '20253Q', '20252Q', '20251Q', '20244Q',
       '20243Q', '20242Q', '20241Q', '20234Q',
       ...
       '19922Q', '19921Q', '19914Q', '19913Q', '19912Q', '19911Q', '19904Q',
       '19903Q', '19902Q', '19901Q'],
      dtype='object', length=145)


In [29]:
# Convert quarterly RPI to annual
df_long = rpi.melt(id_vars=['DataSeries'], var_name='Quarter', value_name='RPI')
df_long['year'] = df_long['Quarter'].str[:4]
df_long['RPI'] = pd.to_numeric(df_long['RPI'], errors='coerce')
annual_rpi = df_long.groupby('year')['RPI'].mean().reset_index()
annual_rpi.columns = ['year', 'average_rpi']

print(annual_rpi.head())

   year  average_rpi
0  1990       24.600
1  1991       25.175
2  1992       27.450
3  1993       41.625
4  1994       52.875


In [43]:
# Merge data
resale['year'] = resale['year'].astype(int)
annual_rpi['year'] = annual_rpi['year'].astype(int)

df_merged = pd.merge(resale, annual_rpi, on='year', how='left')

# Check for missing years
missing_years = df_merged[df_merged['average_rpi'].isnull()]['year'].unique()
if len(missing_years) > 0:
    print(f"The following years are missing RPI values: \n{sorted(list(missing_years))}")
else:
    print("No missing years found.")

The following years are missing RPI values: 
[2026]


In [49]:
# Drop rows where 'Average_RPI' is NaN (the missing years)
df = df_merged.dropna(subset=['average_rpi'])
df = df.reset_index(drop=True)
# Verify that the missing years are gone
remaining_missing = df['average_rpi'].isnull().sum()
print(f"Remaining rows with missing RPI: {remaining_missing}")

Remaining rows with missing RPI: 0


In [51]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, mean_absolute_error

In [55]:
# Create adjusted price column
df['adj_resale_price'] = (df['resale_price'] / df['average_rpi']) * 100
print(df[['year', 'resale_price', 'average_rpi', 'adj_resale_price']].head())

   year  resale_price  average_rpi  adj_resale_price
0  1990        9000.0         24.6      36585.365854
1  1990        6000.0         24.6      24390.243902
2  1990        8000.0         24.6      32520.325203
3  1990        6000.0         24.6      24390.243902
4  1990       47200.0         24.6     191869.918699


In [61]:
# Feature Selection
features = ['floor_area_sqm', 'lease_commence_date', 'avg_storey', 'town', 'flat_type', 'remaining_lease']
X = df[features]
y = df['adj_resale_price']

# One-Hot Encoding
X = pd.get_dummies(X, columns=['town', 'flat_type'], drop_first=True)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Ridge with Cross-Validation
ridge_adj = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0])
ridge_adj.fit(X_train_scaled, y_train)

print(f"R-squared (Adjusted Model): {ridge_adj.score(X_test_scaled, y_test):.4f}")

# Create a DataFrame for the coefficients
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': ridge_adj.coef_
})

# Add a column for 'Absolute Value' to easily find the most impactful features
coef_df['Abs_Value'] = coef_df['Coefficient'].abs()

# Sort by Absolute Value (descending)
coef_df = coef_df.sort_values(by='Abs_Value', ascending=False).drop(columns='Abs_Value')

# Display the top 20 most influential features
print("Top Drivers of Adjusted Resale Price:")
print(coef_df.head(20).to_string(index=False))

R-squared (Adjusted Model): 0.8178
Top Drivers of Adjusted Resale Price:
            Feature   Coefficient
     floor_area_sqm  78618.966052
   flat_type_5-room  44312.228573
flat_type_Executive  39619.087656
   flat_type_4-room  39478.591970
   flat_type_3-room  27029.332782
     town_Woodlands -26273.398315
   town_Jurong West -21105.025691
lease_commence_date  20121.362785
 town_Choa Chu Kang -18550.628609
      town_Sengkang -17265.361563
 town_Bukit Panjang -15621.826897
        town_Yishun -15204.625200
         avg_storey  14558.318426
     town_Sembawang -14254.763320
    remaining_lease  13663.971099
       town_Punggol -12425.740943
   town_Bukit Batok -10068.407036
   town_Bukit Merah   9798.332772
     town_Pasir Ris  -9533.038565
       town_Hougang  -9437.956816
